<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row is one **page-level observation** for a content item in the starter dataset. The contract is for a single snapshot per page, where the available signals describe recent performance and content characteristics. The data is not a page-day panel; it is a cross-sectional page view with historical aggregates and a later decline label.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Unique content_id rows:", df["content_id"].nunique())
print("Duplicate rows by content_id:", df["content_id"].duplicated().sum())
print("Sample columns:", list(df.columns[:10]))

Rows: 30000
Unique content_id rows: 30000
Duplicate rows by content_id: 0
Sample columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


## 2. Fields: feature / label / context / excluded

**Features:** page-level signals such as impressions, clicks, sessions, CTR, average position, content age, word count, character count, and engagement metrics.

**Label / proxy:** the declining-label outcome derived from the trend direction field.

**Context:** identifiers such as content_id and client_id, used for grouping and auditing rather than as training features.

**Excluded:** private identifiers, client names, domains, URLs, page titles, and keyword text should remain excluded to preserve privacy and avoid leakage.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

feature_cols = ["days_with_impressions", "avg_position", "ctr", "word_count", "char_count", "content_age_days"]
label_col = "trend_direction"
context_cols = ["content_id", "client_id"]

print("Feature columns:", feature_cols)
print("Label column:", label_col)
print("Context columns:", context_cols)
print("Missing values by feature column:")
print(df[feature_cols].isna().sum().to_string())

Feature columns: ['days_with_impressions', 'avg_position', 'ctr', 'word_count', 'char_count', 'content_age_days']
Label column: trend_direction
Context columns: ['content_id', 'client_id']
Missing values by feature column:
days_with_impressions       0
avg_position                0
ctr                         0
word_count               7699
char_count               7699
content_age_days            0


## 3. Verify it with queries (grain, counts, missing values, windows)

The contract above is only useful if it is checked. The code below verifies the row grain, the size of the dataset, the presence of missing values in the planned feature columns, and the fact that the label is derived from a later observed trend signal.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

if "is_declining_label" not in df.columns:
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Rows per content_id counts:")
print(df.groupby("content_id").size().value_counts().head().to_string())

print("\nLabel counts:")
print(df["is_declining_label"].value_counts().to_dict())

print("\nMissingness rate for planned feature columns:")
print((df[["days_with_impressions", "avg_position", "ctr", "word_count", "char_count", "content_age_days"]].isna().mean() * 100).round(2).to_string())

Rows per content_id counts:
1    30000

Label counts:
{1: 16262, 0: 13738}

Missingness rate for planned feature columns:
days_with_impressions     0.00
avg_position              0.00
ctr                       0.00
word_count               25.66
char_count               25.66
content_age_days          0.00


## 4. Data limits

This dataset cannot tell us causality or true long-run performance history. It also cannot recover missing earlier periods for pages that were created later, and it may contain uneven coverage across clients or content types. The contract therefore supports a decision-support ranking workflow, not a claim that we fully understand the underlying causal drivers.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Clients represented:", df["client_id"].nunique())
print("Content type distribution:")
print(df["content_type"].value_counts().head().to_string())
print("\nHistorical coverage window for content_age_days:")
print(df["content_age_days"].describe().to_string())

Clients represented: 32
Content type distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697

Historical coverage window for content_age_days:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.